# 04 — Merge BOM + Trend Drivers\n\nMap bottom-up (BOM) and top-down (Trend) driver lists onto each other.\n- **Match** (both sources) → confidence: high, tag: validated\n- **BOM-only** → confidence: medium, tag: incremental\n- **Trend-only** → confidence: low, tag: speculative/disruptive

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from src.llm import safe_chat_json
from src.traceability import TechDriver, DriverOrigin, DriverConfidence
from src import prompts

# load BOM drivers
with open("../data/outputs/bom_state.json") as f:
    bom_state = json.load(f)
bom_drivers = [TechDriver(**d) for d in bom_state["bom_drivers"]]

# load Trend drivers
with open("../data/outputs/trend_state.json") as f:
    trend_state = json.load(f)
trend_drivers = [TechDriver(**d) for d in trend_state["trend_drivers"]]

print(f"BOM drivers: {len(bom_drivers)}")
print(f"Trend drivers: {len(trend_drivers)}")

BOM drivers: 21
Trend drivers: 24


In [2]:
# format driver lists for the merge prompt
bom_list = "\n".join([f"- ID: {d.id} | Name: {d.name} | Desc: {d.description[:150]}" for d in bom_drivers])
trend_list = "\n".join([f"- ID: {d.id} | Name: {d.name} | Desc: {d.description[:150]}" for d in trend_drivers])

prompt = prompts.MERGE_DRIVERS.format(bom_drivers=bom_list, trend_drivers=trend_list)
merge_result = safe_chat_json(prompt, system="You are merging two technology driver lists for regulatory frequency monitoring.")

# validate returned IDs exist
bom_ids = {d.id for d in bom_drivers}
trend_ids = {d.id for d in trend_drivers}
matched_bom_ids = set()
matched_trend_ids = set()

print("=== MATCHES (both sources → validated) ===")
valid_matches = []
for m in merge_result.get("matches", []):
    bid = m.get("bom_driver_id")
    tid = m.get("trend_driver_id")
    if not bid or bid not in bom_ids:
        print(f"  ⚠ BOM ID '{str(bid)[:12]}' not found — skipping match")
        continue
    if not tid or tid not in trend_ids:
        print(f"  ⚠ Trend ID '{str(tid)[:12]}' not found — skipping match")
        continue
    valid_matches.append(m)
    matched_bom_ids.add(bid)
    matched_trend_ids.add(tid)
    print(f"  ✓ '{m.get('unified_name', '?')}'")
    print(f"    Reason: {m.get('reasoning', '')[:100]}")
    print()
merge_result["matches"] = valid_matches

# any BOM/Trend IDs that weren't matched and weren't listed as solo
returned_bom_only = set(merge_result.get("bom_only", [])) & bom_ids
returned_trend_only = set(merge_result.get("trend_only", [])) & trend_ids

# add any drivers the LLM forgot to categorize
forgotten_bom = bom_ids - matched_bom_ids - returned_bom_only
forgotten_trend = trend_ids - matched_trend_ids - returned_trend_only
if forgotten_bom:
    print(f"⚠ {len(forgotten_bom)} BOM drivers not categorized by LLM — adding as BOM-only")
    returned_bom_only |= forgotten_bom
if forgotten_trend:
    print(f"⚠ {len(forgotten_trend)} Trend drivers not categorized by LLM — adding as Trend-only")
    returned_trend_only |= forgotten_trend

merge_result["bom_only"] = list(returned_bom_only)
merge_result["trend_only"] = list(returned_trend_only)

print(f"\n=== BOM-ONLY ({len(merge_result['bom_only'])} drivers) ===")
for bid in merge_result["bom_only"]:
    d = next((x for x in bom_drivers if x.id == bid), None)
    if d:
        print(f"  • {d.name}")

print(f"\n=== TREND-ONLY ({len(merge_result['trend_only'])} drivers) ===")
for tid in merge_result["trend_only"]:
    d = next((x for x in trend_drivers if x.id == tid), None)
    if d:
        print(f"  • {d.name}")

=== MATCHES (both sources → validated) ===
  ⚠ Trend ID 'None' not found — skipping match
  ⚠ Trend ID 'None' not found — skipping match
  ⚠ Trend ID 'None' not found — skipping match
  ⚠ Trend ID 'None' not found — skipping match
  ⚠ Trend ID 'None' not found — skipping match
  ⚠ Trend ID '8e2d82ee6c91' not found — skipping match
  ⚠ Trend ID 'b45ed0c6c1ef' not found — skipping match
  ⚠ Trend ID 'None' not found — skipping match
  ⚠ Trend ID 'None' not found — skipping match
  ⚠ Trend ID 'None' not found — skipping match
  ⚠ Trend ID 'None' not found — skipping match
  ⚠ Trend ID 'None' not found — skipping match
  ⚠ Trend ID 'None' not found — skipping match
  ⚠ Trend ID 'None' not found — skipping match
  ⚠ Trend ID 'None' not found — skipping match
  ⚠ Trend ID 'None' not found — skipping match
  ⚠ Trend ID 'None' not found — skipping match
  ✓ 'Real-Time Bandwidth Capture and Photonic Signal Processing for Ultra-Wideband Receivers'
    Reason: Real-time bandwidth capture up to 2 

In [3]:
# build unified driver list with confidence tags
unified_drivers: list[TechDriver] = []
bom_by_id = {d.id: d for d in bom_drivers}
trend_by_id = {d.id: d for d in trend_drivers}

# matched drivers → high confidence
for m in merge_result.get("matches", []):
    bom_d = bom_by_id.get(m["bom_driver_id"])
    trend_d = trend_by_id.get(m["trend_driver_id"])
    if bom_d and trend_d:
        merged_sources = list(set(bom_d.source_chunk_ids + trend_d.source_chunk_ids))
        driver = TechDriver(
            name=m["unified_name"],
            description=f"BOM: {bom_d.description[:200]} | Trend: {trend_d.description[:200]}",
            origin=DriverOrigin.BOTH,
            confidence=DriverConfidence.HIGH,
            bom_node_id=bom_d.bom_node_id,
            source_chunk_ids=merged_sources,
            merge_reasoning=m.get("reasoning", ""),
        )
        unified_drivers.append(driver)

# BOM-only → medium confidence
for bid in merge_result.get("bom_only", []):
    d = bom_by_id.get(bid)
    if d:
        d.confidence = DriverConfidence.MEDIUM
        unified_drivers.append(d)

# Trend-only → low confidence (speculative)
for tid in merge_result.get("trend_only", []):
    d = trend_by_id.get(tid)
    if d:
        d.confidence = DriverConfidence.LOW
        unified_drivers.append(d)

print(f"=== Unified Driver List (before consolidation): {len(unified_drivers)} drivers ===")

=== Unified Driver List (before consolidation): 41 drivers ===


## Consolidate Near-Duplicates

Use embedding cosine similarity to find and merge near-duplicate drivers.
This prevents the CIB matrix from exploding with redundant pairs.

In [4]:
# Exact-name-match dedup — catches duplicates that cosine similarity might miss
from collections import Counter

before_count = len(unified_drivers)
exact_seen: dict[str, int] = {}
for i, d in enumerate(unified_drivers):
    key = d.name.lower().strip()
    if key in exact_seen:
        first = unified_drivers[exact_seen[key]]
        first.source_chunk_ids = list(set(first.source_chunk_ids + d.source_chunk_ids))
        unified_drivers[i] = None
        print(f"  Dedup: '{d.name}' merged into existing")
    else:
        exact_seen[key] = i
unified_drivers = [d for d in unified_drivers if d is not None]

dupes_removed = before_count - len(unified_drivers)
print(f"Exact-name dedup: {before_count} → {len(unified_drivers)} ({dupes_removed} removed)")

# verify no exact dupes remain
name_counts = Counter(d.name.lower().strip() for d in unified_drivers)
remaining = {n: c for n, c in name_counts.items() if c > 1}
assert not remaining, f"Duplicate drivers remain: {remaining}"

  Dedup: 'Angle-of-Arrival Error Correction' merged into existing
  Dedup: 'Angle-of-Arrival Error Correction' merged into existing
  Dedup: 'Angle-of-Arrival Error Correction' merged into existing
  Dedup: 'Filtering Architecture' merged into existing
  Dedup: 'Angle-of-Arrival Error Correction' merged into existing
  Dedup: 'Tunable Bandpass Filters' merged into existing
  Dedup: 'Internal GNSS Module' merged into existing
Exact-name dedup: 41 → 34 (7 removed)


In [5]:
import numpy as np
from src.llm import embed

SIMILARITY_THRESHOLD = 0.78

texts = [f"{d.name}: {d.description[:150]}" for d in unified_drivers]
embeddings = np.array(embed(texts))

norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
sim_matrix = (embeddings @ embeddings.T) / (norms @ norms.T + 1e-10)

merged_indices = set()
consolidated: list[TechDriver] = []

for i in range(len(unified_drivers)):
    if i in merged_indices:
        continue
    cluster = [i]
    for j in range(i + 1, len(unified_drivers)):
        if j in merged_indices:
            continue
        if sim_matrix[i][j] > SIMILARITY_THRESHOLD:
            cluster.append(j)
            merged_indices.add(j)

    # keep driver with highest confidence, then most sources
    conf_rank = {"high": 3, "medium": 2, "low": 1}
    best_idx = max(cluster, key=lambda idx: (
        conf_rank.get(unified_drivers[idx].confidence.value, 0),
        len(unified_drivers[idx].source_chunk_ids),
    ))
    rep = unified_drivers[best_idx]

    all_sources = set()
    for idx in cluster:
        all_sources.update(unified_drivers[idx].source_chunk_ids)
    rep.source_chunk_ids = list(all_sources)

    consolidated.append(rep)
    if len(cluster) > 1:
        names = [unified_drivers[idx].name[:40] for idx in cluster]
        print(f"  Merged {len(cluster)}: {names} → {rep.name[:50]}")

print(f"\nConsolidated: {len(unified_drivers)} → {len(consolidated)} drivers")
unified_drivers = consolidated

  Merged 3: ['Time Difference of Arrival (TDOA) Proces', 'Time Difference of Arrival (TDOA) Functi', 'Time Difference of Arrival (TDOA) Measur'] → Time Difference of Arrival (TDOA) Processing
  Merged 2: ['Real-Time Bandwidth Processing (up to 12', 'Real-Time Bandwidth Processing'] → Real-Time Bandwidth Processing (up to 125 MHz)
  Merged 2: ['AI-Driven Dynamic Spectrum Access and Co', 'AI-Driven Spectrum Sensing and Dynamic S'] → AI-Driven Dynamic Spectrum Access and Cognitive Ra
  Merged 2: ['6G and AI-Native Intelligent Networks', '6G and AI-Native Networks'] → 6G and AI-Native Intelligent Networks
  Merged 2: ['Photonic Signal Processing for Ultra-Wid', 'Photonic Signal Processing for Ultra-Wid'] → Photonic Signal Processing for Ultra-Wideband Spec
  Merged 2: ['Cognitive Radio Evolution with Federated', 'Federated Learning for Decentralized Spe'] → Cognitive Radio Evolution with Federated Learning 

Consolidated: 34 → 27 drivers


In [6]:
# save consolidated unified drivers
print(f"=== Final Unified Driver List: {len(unified_drivers)} drivers ===\n")
for d in unified_drivers:
    origin_tag = {"both": "VALIDATED", "bom": "INCREMENTAL", "trend": "SPECULATIVE"}[d.origin.value]
    print(f"  [{d.confidence.value:6s}] [{origin_tag:11s}] {d.name}")

merge_state = {
    "unified_drivers": [d.model_dump(mode="json") for d in unified_drivers],
    "merge_result": merge_result,
}
with open("../data/outputs/merge_state.json", "w") as f:
    json.dump(merge_state, f, indent=2)
print(f"\nSaved {len(unified_drivers)} drivers to merge_state.json")

=== Final Unified Driver List: 27 drivers ===

  [high  ] [VALIDATED  ] Real-Time Bandwidth Capture and Photonic Signal Processing for Ultra-Wideband Receivers
  [high  ] [VALIDATED  ] Digital Downconversion and Evolution of Software-Defined Radio (SDR) Architectures
  [high  ] [VALIDATED  ] I/Q Data Streaming Interface and Edge Computing for Distributed Spectrum Monitoring
  [high  ] [VALIDATED  ] Dynamic Range Enhancement System and Energy-Efficient AI Architectures for Spectrum Monitoring
  [medium] [INCREMENTAL] Angle-of-Arrival Error Correction
  [medium] [INCREMENTAL] Time Difference of Arrival (TDOA) Processing
  [medium] [INCREMENTAL] Tunable Bandpass Filters
  [medium] [INCREMENTAL] TDOA Functionality with Internal GNSS Module
  [medium] [INCREMENTAL] Real-Time Bandwidth Processing (up to 125 MHz)
  [medium] [INCREMENTAL] Filtering Architecture
  [medium] [INCREMENTAL] Internal GNSS Module
  [low   ] [SPECULATIVE] AI-Driven Dynamic Spectrum Access and Cognitive Radio Networks
